# Bayesian Disk Failure Prediction and Risk‑Optimised Maintenance
This notebook demonstrates a workflow for predicting disk failures with uncertainty, building a simple economic model, and optimising the maintenance policy by balancing expected income against downside risk (CVaR).


In [5]:

# !pip install catboost pymoo tqdm
import glob
import pandas as pd
from pathlib import Path

DATA_PATH = Path('/Users/cobeq/Downloads/data_Q4_2024')
files = sorted(DATA_PATH.glob('*.csv'))[-10:]
df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
df=df[df["smart_9_raw"].notna()]
df['date'] = pd.to_datetime(df['date'])
print(f'Loaded {len(df):,} rows from {len(files)} daily files')
df.head()


Loaded 3,046,463 rows from 10 daily files


,date,serial_number,model,capacity_bytes,failure,datacenter,cluster_id,vault_id,pod_id,pod_slot_num,...,smart_250_normalized,smart_250_raw,smart_251_normalized,smart_251_raw,smart_252_normalized,smart_252_raw,smart_254_normalized,smart_254_raw,smart_255_normalized,smart_255_raw
0,2024-12-22,S2ZYJ9GGB01000,ST500LM012 HN,500107862016,0,sac0,0,1028,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-12-22,VKH7A84X,HGST HUH728080ALE600,8001563222016,0,sac0,0,1028,0,22.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024-12-22,ZA106RPJ,ST8000DM002,8001563222016,0,sac0,0,1028,0,28.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2024-12-22,ZA106YX1,ST8000DM002,8001563222016,0,sac0,0,1028,0,23.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024-12-22,ZA106Z5M,ST8000DM002,8001563222016,0,sac0,0,1028,0,39.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:

import pandas as pd
import numpy as np
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.termination import get_termination
from pymoo.optimize import minimize
import matplotlib.pyplot as plt
import warnings, random
warnings.filterwarnings('ignore')


## 1. Load the SMART statistics dataset

In [3]:

# Replace this with the actual loading code for your data
# df = pd.read_csv('smart_stats.csv')
assert 'df' in globals(), "Please load your DataFrame into a variable called `df` before running the next cells."
df.head()


,date,serial_number,model,capacity_bytes,failure,datacenter,cluster_id,vault_id,pod_id,pod_slot_num,...,smart_250_normalized,smart_250_raw,smart_251_normalized,smart_251_raw,smart_252_normalized,smart_252_raw,smart_254_normalized,smart_254_raw,smart_255_normalized,smart_255_raw
0,2024-12-22,S2ZYJ9GGB01000,ST500LM012 HN,500107862016,0,sac0,0,1028,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-12-22,VKH7A84X,HGST HUH728080ALE600,8001563222016,0,sac0,0,1028,0,22.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024-12-22,ZA106RPJ,ST8000DM002,8001563222016,0,sac0,0,1028,0,28.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2024-12-22,ZA106YX1,ST8000DM002,8001563222016,0,sac0,0,1028,0,23.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024-12-22,ZA106Z5M,ST8000DM002,8001563222016,0,sac0,0,1028,0,39.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Split data and train CatBoost model

In [6]:

target = 'failure'
feature_cols = [c for c in df.columns if c != target]
X = df[feature_cols]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

train_pool = Pool(X_train, y_train)
test_pool = Pool(X_test, y_test)

params = dict(
    iterations=800,
    depth=6,
    learning_rate=0.05,
    loss_function='Logloss',
    verbose=False
)

base_model = CatBoostClassifier(**params)
base_model.fit(train_pool, eval_set=test_pool, verbose=200)


CatBoostError: Bad value for num_feature[non_default_doc_idx=0,feature_idx=1]="D5G18HZL": Cannot convert 'D5G18HZL' to float

### 2.1 Posterior sampling via a bootstrap ensemble

In [7]:

def fit_bootstrap_ensemble(pool, n_models=20, random_seed=42):
    ensemble = []
    rng = np.random.default_rng(random_seed)
    for i in range(n_models):
        seed = rng.integers(0, 1e6)
        params_copy = params.copy()
        params_copy['random_seed'] = int(seed)
        # Bootstrap resampling of indices
        idx = rng.choice(pool.num_row(), size=pool.num_row(), replace=True)
        boot_pool = Pool(pool.get_features()[idx], pool.get_label()[idx])
        model = CatBoostClassifier(**params_copy)
        model.fit(boot_pool, verbose=False)
        ensemble.append(model)
    return ensemble

ensemble = fit_bootstrap_ensemble(train_pool, n_models=25)


NameError: name 'train_pool' is not defined

In [ ]:

def posterior_predict_proba(ensemble, pool):
    return np.stack([m.predict_proba(pool)[:,1] for m in ensemble], axis=0)

posterior_test = posterior_predict_proba(ensemble, test_pool)  # shape (n_models, n_disks)
posterior_test.shape


## 3. Economic model and Monte‑Carlo simulation

In [ ]:

# Economic parameters (set these to match your business scenario)
C_FAIL = 5000.0     # Cost incurred when a disk fails in service
C_REPLACE = 200.0   # Preventive replacement cost


In [ ]:

def simulate_income(prob_samples, threshold, n_mc=2000):
    n_models, n_disks = prob_samples.shape
    incomes = []
    for _ in range(n_mc):
        # Sample a posterior model for each disk
        idx = np.random.randint(0, n_models, size=n_disks)
        p = prob_samples[idx, np.arange(n_disks)]
        replace = p > threshold
        cost_replace = replace.sum() * C_REPLACE
        # Failures among disks NOT replaced
        probs_run = p[~replace]
        fails = np.random.rand(len(probs_run)) < probs_run
        cost_fail = fails.sum() * C_FAIL
        incomes.append(-(cost_replace + cost_fail))
    return np.asarray(incomes)


In [ ]:

def cvar(series, alpha=0.95):
    """Conditional Value at Risk of the lower tail (losses)."""
    q = np.quantile(series, 1 - alpha)
    tail = series[series <= q]
    return tail.mean() if len(tail) else q


## 4. Risk–return optimisation with pymoo

In [ ]:

class MaintenanceProblem(ElementwiseProblem):
    def __init__(self, prob_samples, n_mc=2000, alpha=0.95):
        super().__init__(n_var=1, n_obj=2, n_constr=0, xl=np.array([0.0]), xu=np.array([1.0]))
        self.prob_samples = prob_samples
        self.n_mc = n_mc
        self.alpha = alpha

    def _evaluate(self, x, out, *args, **kwargs):
        thr = x[0]
        incomes = simulate_income(self.prob_samples, thr, n_mc=self.n_mc)
        expected_income = incomes.mean()
        cvar_income = cvar(incomes, self.alpha)
        # convert to minimisation objectives
        out["F"] = np.array([-expected_income, -cvar_income])


In [ ]:

problem = MaintenanceProblem(posterior_test, n_mc=2000, alpha=0.95)
algorithm = NSGA2(pop_size=40)
termination = get_termination("n_gen", 60)

result = minimize(problem,
                  algorithm,
                  termination,
                  seed=1,
                  verbose=False)


### 4.1 Pareto front

In [ ]:

F = result.F
plt.scatter(-F[:,0], -F[:,1])
plt.xlabel("Expected income")
plt.ylabel("CVaR (α=0.95)")
plt.title("Income–Risk trade‑off (Pareto front)")
plt.show()


### 4.2 Example maintenance policy

In [ ]:

# Select the solution that maximises expected income
best_idx = np.argmax(-F[:,0])
best_thr = result.X[best_idx][0]
print(f"Suggested replacement probability threshold: {best_thr:.3f}")

incomes_eval = simulate_income(posterior_test, best_thr, n_mc=5000)
print(f"Expected income: {incomes_eval.mean():.2f}")
print(f"CVaR (α=0.95): {cvar(incomes_eval):.2f}")
